# UMAP spike — Altair + polars port

Rebuilds `interactive/src/figures/data/umap/App.svelte` as an Altair chart authored in Python, to evaluate whether Altair can replace SveltePlot for data-driven figures.

> **⚠ Visual quality is unverified.** This notebook was generated by Claude, who cannot render or view images. Every descriptive phrase below about how a variant "looks", "reads", or "matches" is theory about the underlying algorithm, not an observation of the output. **Open the HTML files in a browser and judge the composition, proportions, and fidelity yourself before acting on any variant.** Known file-size / path-count data is reliable; anything aesthetic is not.

**What the original does (Svelte + SveltePlot):**
- Main panel: scatter + 6-level KDE density contours + dashed convex hulls, per class
- Top marginal: histogram bars + KDE line (x-axis distribution per class)
- Right marginal: rotated hist + KDE (y-axis distribution)
- Toggle buttons above the plot: show/hide each class across all layers
- Metrics row below: Wasserstein, energy distance, KDE overlap

**What this spike produces (per variant):**
- `*.html` — standalone interactive HTML (Vega-Embed)
- `*.spec.json` — the Vega-Lite spec JSON
- `*.svg` — static render via `vl-convert`

Run: `uv run --extra notebooks jupyter execute tools/spikes/umap_altair.ipynb` or open in VS Code / JupyterLab.


## 1. Load data and set up


In [1]:
from pathlib import Path
import json

import numpy as np
import polars as pl
import altair as alt
from skimage import measure
import vl_convert as vlc

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / "pyproject.toml").exists():
    REPO = REPO.parent

DATA_PATH = REPO / "interactive" / "src" / "figures" / "data" / "umap" / "data.json"
OUT_DIR = REPO / "tools" / "spikes"
OUT_DIR.mkdir(exist_ok=True)

data = json.loads(DATA_PATH.read_text())
_first = next(iter(data["density"]))
_d0 = data["density"][_first]
print(f"classes:  {list(data['density'].keys())}")
print(f"points:   {len(data['points'])}")
print(f"grid:     {_d0['height']} x {_d0['width']}  (flat list of {len(_d0['grid'])} floats, row-major)")
print(f"bounds:   {data['bounds']}")


classes:  ['Attack', 'Normal']
points:   500
grid:     100 x 100  (flat list of 10000 floats, row-major)
bounds:   {'x1': -13.72, 'y1': -7.0, 'x2': 19.8, 'y2': 18.91}


## 2. Palette

Replicate `resolve(key).stroke` from `interactive/src/lib/flow/palette.ts`. Values here are the Observable 10 hex codes that `styles.yml` uses — for the spike we hardcode instead of parsing the YAML, since the question is whether Altair works, not whether Python can read YAML.


In [2]:
PALETTE = {
    "normal":    "#59A14F",
    "attack":    "#E15759",
    "gat":       "#4E79A7",
    "dqn":       "#F28E2B",
    "data":      "#76B7B2",
    "attention": "#B07AA1",
    "kd":        "#EDC948",
}
PALETTE_ORDER = list(PALETTE.keys())

attack_types = sorted({p["attack_type"] for p in data["points"]})

def color_for(t: str, i: int) -> str:
    key = t.lower() if t.lower() in PALETTE else PALETTE_ORDER[i % len(PALETTE_ORDER)]
    return PALETTE[key]

color_map = {t: color_for(t, i) for i, t in enumerate(attack_types)}
color_map


{'Attack': '#E15759', 'Normal': '#59A14F'}

## 3. Build polars DataFrames (points, contours, hulls, marginals)


In [3]:
points_df = pl.DataFrame(data["points"])
points_df.head()


x,y,label,attack_type
f64,f64,i64,str
-7.389536,3.41373,1,"""Attack"""
9.241927,7.789581,0,"""Normal"""
16.835016,-0.165586,0,"""Normal"""
-11.884244,4.44864,1,"""Attack"""
9.665522,7.989795,0,"""Normal"""


In [4]:
# Contours: run marching-squares per level per class, map grid indices back to data coords.
# (SveltePlot's <Contour> mark does this in-browser; in Altair we precompute polygons.)
x1 = data["bounds"]["x1"]
y1 = data["bounds"]["y1"]
x2 = data["bounds"]["x2"]
y2 = data["bounds"]["y2"]

N_LEVELS = 6
# Skip level 0.0 (entire grid) — matches the Svelte version's opacity ramp starting > 0
levels = np.linspace(0.0, 1.0, N_LEVELS)[1:]

contour_rows = []
for t, spec in data["density"].items():
    W, H = spec["width"], spec["height"]
    grid = np.asarray(spec["grid"], dtype=float).reshape((H, W))
    gmax = grid.max()
    g = grid / gmax if gmax > 0 else grid
    for li, level in enumerate(levels):
        polys = measure.find_contours(g, level)
        for ci, poly in enumerate(polys):
            xs = x1 + (poly[:, 1] / (W - 1)) * (x2 - x1)
            ys = y1 + (poly[:, 0] / (H - 1)) * (y2 - y1)
            poly_id = f"{t}_{li}_{ci}"
            for k, (x, y) in enumerate(zip(xs, ys)):
                contour_rows.append({
                    "attack_type": t,
                    "level": float(level),
                    "level_idx": int(li),
                    "poly_id": poly_id,
                    "order": int(k),
                    "x": float(x),
                    "y": float(y),
                })

contours_df = pl.DataFrame(contour_rows)
print(f"{contours_df.height} contour vertices across {contours_df['poly_id'].n_unique()} polygons")
contours_df.head()


1269 contour vertices across 11 polygons


attack_type,level,level_idx,poly_id,order,x,y
str,f64,i64,str,i64,f64,f64
"""Attack""",0.2,0,"""Attack_0_0""",0,-3.90101,7.37795
"""Attack""",0.2,0,"""Attack_0_0""",1,-4.239596,7.304358
"""Attack""",0.2,0,"""Attack_0_0""",2,-4.578182,7.161418
"""Attack""",0.2,0,"""Attack_0_0""",3,-4.629597,7.132727
"""Attack""",0.2,0,"""Attack_0_0""",4,-4.916768,6.98364


In [5]:
hull_rows = []
for t, poly in data["hulls"].items():
    for k, p in enumerate(poly):
        hull_rows.append({"attack_type": t, "order": k, "x": p["x"], "y": p["y"]})
hulls_df = pl.DataFrame(hull_rows)
hulls_df.head()


attack_type,order,x,y
str,i64,f64,f64
"""Attack""",0,-12.197,4.477
"""Attack""",1,2.3,-2.509
"""Attack""",2,1.256,17.538
"""Attack""",3,1.229,17.621
"""Attack""",4,1.222,17.634


In [6]:
def build_hist(axis: str) -> pl.DataFrame:
    rows = []
    for t, spec in data["marginals"].items():
        ax = spec[axis]
        edges = ax["bin_edges"]
        for i, d in enumerate(ax["bin_density"]):
            rows.append({"attack_type": t, "start": edges[i], "end": edges[i + 1], "density": d})
    return pl.DataFrame(rows)

def build_kde(axis: str) -> pl.DataFrame:
    rows = []
    for t, spec in data["marginals"].items():
        ax = spec[axis]
        for v, d in zip(ax["kde_values"], ax["kde_density"]):
            rows.append({"attack_type": t, "v": v, "d": d})
    return pl.DataFrame(rows)

hist_x, hist_y = build_hist("x"), build_hist("y")
kde_x, kde_y = build_kde("x"), build_kde("y")
print(f"hist_x {hist_x.shape}  hist_y {hist_y.shape}  kde_x {kde_x.shape}  kde_y {kde_y.shape}")


hist_x (60, 4)  hist_y (60, 4)  kde_x (200, 3)  kde_y (200, 3)


## 4. Interactivity

The Svelte version has custom toggle buttons above the plot. Altair gives us the same behavior for free via **legend-bound point selection** — clicking a legend entry toggles that class across *every* layer (contours, hulls, scatter, marginals) as long as each layer conditions on the same selection.


In [7]:
color_scale = alt.Scale(domain=list(color_map.keys()), range=list(color_map.values()))
selection = alt.selection_point(fields=["attack_type"], bind="legend")

# Shared opacity expression: visible when the class is selected (or no selection active)
# Scatter/hulls use binary visible/hidden; contours ramp by level for the banded effect.
def on_off(visible: float = 0.6, hidden: float = 0.0):
    return alt.condition(selection, alt.value(visible), alt.value(hidden))


## 5. Main panel: scatter + hulls + contours


In [8]:
base_x = alt.X("x:Q", scale=alt.Scale(domain=[x1, x2]), axis=alt.Axis(title="UMAP 1"))
base_y = alt.Y("y:Q", scale=alt.Scale(domain=[y1, y2]), axis=alt.Axis(title="UMAP 2"))

# Contour lines: one stroked path per (class, level, polygon). Opacity ramps with level.
contour_chart = (
    alt.Chart(contours_df)
    .mark_line(strokeWidth=0.5)
    .encode(
        x=base_x,
        y=base_y,
        color=alt.Color("attack_type:N", scale=color_scale, legend=alt.Legend(title=None, orient="top")),
        detail="poly_id:N",
        order="order:Q",
        opacity=alt.condition(
            selection,
            alt.Opacity("level:Q", scale=alt.Scale(domain=[float(levels[0]), float(levels[-1])], range=[0.05, 0.35]), legend=None),
            alt.value(0.0),
        ),
    )
)

hull_chart = (
    alt.Chart(hulls_df)
    .mark_line(strokeDash=[6, 3], strokeWidth=1.5)
    .encode(
        x="x:Q",
        y="y:Q",
        color=alt.Color("attack_type:N", scale=color_scale, legend=None),
        detail="attack_type:N",
        order="order:Q",
        opacity=on_off(0.5),
    )
)

scatter = (
    alt.Chart(points_df)
    .mark_circle(size=10)
    .encode(
        x="x:Q",
        y="y:Q",
        color=alt.Color("attack_type:N", scale=color_scale, legend=None),
        opacity=on_off(0.6),
    )
)

main = (contour_chart + hull_chart + scatter).properties(width=400, height=400)
main


alt.LayerChart(...)

## 6. Marginals (top + right)


In [9]:
top_hist = (
    alt.Chart(hist_x)
    .mark_rect()
    .encode(
        x=alt.X("start:Q", scale=alt.Scale(domain=[x1, x2]), axis=None),
        x2="end:Q",
        y=alt.Y("density:Q", axis=None),
        color=alt.Color("attack_type:N", scale=color_scale, legend=None),
        opacity=on_off(0.25),
    )
)

top_kde = (
    alt.Chart(kde_x)
    .mark_line(strokeWidth=1.5)
    .encode(
        x=alt.X("v:Q", scale=alt.Scale(domain=[x1, x2]), axis=None),
        y=alt.Y("d:Q", axis=None),
        color=alt.Color("attack_type:N", scale=color_scale, legend=None),
        detail="attack_type:N",
        opacity=on_off(1.0),
    )
)

top_marginal = (top_hist + top_kde).properties(width=400, height=60)

right_hist = (
    alt.Chart(hist_y)
    .mark_rect()
    .encode(
        y=alt.Y("start:Q", scale=alt.Scale(domain=[y1, y2]), axis=None),
        y2="end:Q",
        x=alt.X("density:Q", axis=None),
        color=alt.Color("attack_type:N", scale=color_scale, legend=None),
        opacity=on_off(0.25),
    )
)

right_kde = (
    alt.Chart(kde_y)
    .mark_line(strokeWidth=1.5)
    .encode(
        y=alt.Y("v:Q", scale=alt.Scale(domain=[y1, y2]), axis=None),
        x=alt.X("d:Q", axis=None),
        color=alt.Color("attack_type:N", scale=color_scale, legend=None),
        detail="attack_type:N",
        opacity=on_off(1.0),
    )
)

right_marginal = (right_hist + right_kde).properties(width=60, height=400)

top_marginal | right_marginal  # quick preview (side-by-side)


alt.HConcatChart(...)

## 7. Compose the 2×2 grid


In [10]:
# Empty spacer for the top-right corner so row heights line up
spacer = alt.Chart(pl.DataFrame({"_": [0]})).mark_text(text="").properties(width=60, height=60)

m = data["metrics"]
subtitle = (
    f"Wasserstein {m['wasserstein_2d']}  ·  "
    f"Energy dist {m['energy_distance']}  ·  "
    f"KDE overlap {m['overlap_integral']:.1e}"
)

chart = (
    alt.vconcat(
        alt.hconcat(top_marginal, spacer, spacing=5),
        alt.hconcat(main, right_marginal, spacing=5),
        spacing=5,
    )
    .add_params(selection)
    .properties(
        title=alt.TitleParams(
            text="UMAP Projections of GAT Embeddings",
            subtitle=subtitle,
            anchor="start",
        )
    )
    .configure_view(stroke=None)
)
chart


/mnt/c/Users/User1/Documents/GitHub/kd-gat-paper/.venv/lib/python3.12/site-packages/altair/vegalite/v6/api.py:5264: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  subcharts = [subchart.copy() for subchart in subcharts]
/tmp/ipykernel_24514/2581880572.py:18: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  .properties(


alt.VConcatChart(...)

## 8. Save outputs (HTML / spec JSON / SVG)


In [11]:
HTML_OUT = OUT_DIR / "umap_altair.html"
SPEC_OUT = OUT_DIR / "umap_altair.spec.json"
SVG_OUT  = OUT_DIR / "umap_altair.svg"

chart.save(str(HTML_OUT))
SPEC_OUT.write_text(json.dumps(chart.to_dict(), indent=2))

svg = vlc.vegalite_to_svg(json.dumps(chart.to_dict()))
SVG_OUT.write_text(svg)

def kb(p: Path) -> str:
    return f"{p.stat().st_size / 1024:7.1f} KB"

print(f"HTML: {HTML_OUT.relative_to(REPO)}   {kb(HTML_OUT)}")
print(f"SPEC: {SPEC_OUT.relative_to(REPO)}   {kb(SPEC_OUT)}")
print(f"SVG : {SVG_OUT.relative_to(REPO)}   {kb(SVG_OUT)}")


HTML: tools/spikes/umap_altair.html     270.3 KB
SPEC: tools/spikes/umap_altair.spec.json     408.1 KB
SVG : tools/spikes/umap_altair.svg     223.6 KB


## 9. Variant B — filled density bands (annular, matplotlib-style)

Variant A emits stroked isolines via `mark_line`. Variant B computes true annular bands: for each level `N`, take the region `{density >= level_N}` and subtract `{density >= level_{N+1}}` to get an annulus, rendered with `mark_area`. This is the rendering semantics matplotlib's `contourf` produces (one polygon set per inter-level gap).

`skimage.measure.find_contours` returns every isoline as a separate path (nested rings of one level are separate entries). We hand those to `shapely.Polygon(...).buffer(0)` to fix winding, `unary_union` per level to merge islands, then `difference` adjacent levels. The result feeds Altair's `mark_area` via a `DataFrame` with one row per polygon vertex.

> ⚠ Whether this actually looks like what we want is a visual judgment — open the resulting HTML to decide.


In [12]:
from shapely.geometry import Polygon, MultiPolygon
from shapely.ops import unary_union

def _polys_at_level(grid_norm: np.ndarray, level: float) -> list[Polygon]:
    """Return valid Shapely polygons enclosing the region {grid >= level}.

    `find_contours` yields both outer boundaries and holes at the same level when
    the region is simply not simply-connected; we collect them all as polygons and
    let `unary_union` merge / hole-out appropriately.
    """
    polys = []
    for path in measure.find_contours(grid_norm, level):
        if len(path) < 4:
            continue
        # (row, col) -> (col, row) ordering; still in grid index space
        ring = [(float(c), float(r)) for r, c in path]
        try:
            p = Polygon(ring).buffer(0)  # buffer(0) cleans self-intersections
        except Exception:
            continue
        if p.is_empty:
            continue
        polys.append(p)
    return polys

def _grid_to_data(xy: tuple[float, float], W: int, H: int) -> tuple[float, float]:
    gx, gy = xy
    return (
        x1 + (gx / (W - 1)) * (x2 - x1),
        y1 + (gy / (H - 1)) * (y2 - y1),
    )

def _iter_polygons(geom):
    if isinstance(geom, Polygon):
        if not geom.is_empty:
            yield geom
    elif isinstance(geom, MultiPolygon):
        for g in geom.geoms:
            if not g.is_empty:
                yield g

# For annular bands, include an extra upper level so top band = {level_last <= d < 1.0}
band_levels = list(levels) + [1.0]

band_rows = []
for t, spec in data["density"].items():
    W, H = spec["width"], spec["height"]
    grid = np.asarray(spec["grid"], dtype=float).reshape((H, W))
    gmax = grid.max()
    g = grid / gmax if gmax > 0 else grid

    # Build a union-of-polygons per level threshold
    level_geoms = {}
    for lv in band_levels:
        polys = _polys_at_level(g, lv)
        level_geoms[lv] = unary_union(polys) if polys else Polygon()

    # Annular bands: region(level_i) - region(level_{i+1})
    for li in range(len(band_levels) - 1):
        lv_lo = band_levels[li]
        lv_hi = band_levels[li + 1]
        outer = level_geoms[lv_lo]
        inner = level_geoms[lv_hi]
        if outer.is_empty:
            continue
        band = outer.difference(inner) if not inner.is_empty else outer
        if band.is_empty:
            continue

        for pi, poly in enumerate(_iter_polygons(band)):
            # Exterior ring only (mark_area can't render holes; visually OK because
            # inner band's own area will cover the hole region).
            coords = list(poly.exterior.coords)
            poly_id = f"{t}_band{li}_{pi}"
            for k, (gx, gy) in enumerate(coords):
                dx, dy = _grid_to_data((gx, gy), W, H)
                band_rows.append({
                    "attack_type": t,
                    "level_idx": li,
                    "level": float(lv_lo),
                    "poly_id": poly_id,
                    "order": k,
                    "x": dx,
                    "y": dy,
                })

bands_df = pl.DataFrame(band_rows)
print(f"{bands_df.height} band vertices across {bands_df['poly_id'].n_unique()} annular polygons")
bands_df.head()


1402 band vertices across 11 annular polygons


attack_type,level_idx,level,poly_id,order,x,y
str,i64,f64,str,i64,f64,f64
"""Attack""",0,0.2,"""Attack_band0_0""",0,-3.569387,6.87101
"""Attack""",0,0.2,"""Attack_band0_0""",1,-3.628642,6.609293
"""Attack""",0,0.2,"""Attack_band0_0""",2,-3.710587,6.347576
"""Attack""",0,0.2,"""Attack_band0_0""",3,-3.809252,6.085859
"""Attack""",0,0.2,"""Attack_band0_0""",4,-3.90101,5.888336


In [13]:
# Variant B main panel: filled bands + hulls + scatter
# Opacity ramps with level_idx (inner bands = denser = more opaque).
band_opacity_range = [0.06, 0.28]
band_idx_domain = [0, len(band_levels) - 2]

band_chart = (
    alt.Chart(bands_df)
    .mark_area(interpolate="linear")
    .encode(
        x=base_x,
        y=base_y,
        color=alt.Color("attack_type:N", scale=color_scale, legend=alt.Legend(title=None, orient="top")),
        detail="poly_id:N",
        order="order:Q",
        opacity=alt.condition(
            selection,
            alt.Opacity("level_idx:Q", scale=alt.Scale(domain=band_idx_domain, range=band_opacity_range), legend=None),
            alt.value(0.0),
        ),
    )
)

main_b = (band_chart + hull_chart + scatter).properties(width=400, height=400)
main_b


alt.LayerChart(...)

In [14]:
chart_b = (
    alt.vconcat(
        alt.hconcat(top_marginal, spacer, spacing=5),
        alt.hconcat(main_b, right_marginal, spacing=5),
        spacing=5,
    )
    .add_params(selection)
    .properties(
        title=alt.TitleParams(
            text="UMAP Projections of GAT Embeddings  ·  filled bands",
            subtitle=subtitle,
            anchor="start",
        )
    )
    .configure_view(stroke=None)
)

HTML_B = OUT_DIR / "umap_altair_filled.html"
SPEC_B = OUT_DIR / "umap_altair_filled.spec.json"
SVG_B  = OUT_DIR / "umap_altair_filled.svg"

chart_b.save(str(HTML_B))
SPEC_B.write_text(json.dumps(chart_b.to_dict(), indent=2))
SVG_B.write_text(vlc.vegalite_to_svg(json.dumps(chart_b.to_dict())))

print(f"HTML: {HTML_B.relative_to(REPO)}   {kb(HTML_B)}")
print(f"SPEC: {SPEC_B.relative_to(REPO)}   {kb(SPEC_B)}")
print(f"SVG : {SVG_B.relative_to(REPO)}   {kb(SVG_B)}")
chart_b


/tmp/ipykernel_24514/377693150.py:8: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  .properties(


HTML: tools/spikes/umap_altair_filled.html     294.7 KB
SPEC: tools/spikes/umap_altair_filled.spec.json     441.6 KB
SVG : tools/spikes/umap_altair_filled.svg   37260.5 KB


alt.VConcatChart(...)

## 10. Variant C — `mark_rect` heatmap

Skip marching-squares entirely: render each cell of the 100×100 density grid as a tiny rectangle and encode the (normalized) density value as opacity. Simpler, more Altair-native, but loses the banded / isoline look. The trade-off is spec size — 10,000 cells × 7 classes = 70,000 rows of rectangles, pre-thresholding.

To keep the spec reasonable we drop cells where the normalized density is below a small threshold (they're invisible anyway).


In [15]:
HEATMAP_THRESHOLD = 0.05  # drop cells with normalized density below this
cell_w = (x2 - x1) / 100
cell_h = (y2 - y1) / 100

heatmap_rows = []
for t, spec in data["density"].items():
    W, H = spec["width"], spec["height"]
    grid = np.asarray(spec["grid"], dtype=float).reshape((H, W))
    gmax = grid.max()
    if gmax <= 0:
        continue
    g = grid / gmax
    # Vectorized: build coordinate grids, then mask
    rows, cols = np.indices(g.shape)
    xs = x1 + (cols / (W - 1)) * (x2 - x1)
    ys = y1 + (rows / (H - 1)) * (y2 - y1)
    mask = g >= HEATMAP_THRESHOLD
    for gx, gy, gv in zip(xs[mask].ravel(), ys[mask].ravel(), g[mask].ravel()):
        heatmap_rows.append({
            "attack_type": t,
            "x": float(gx),
            "y": float(gy),
            "x2": float(gx + cell_w),
            "y2": float(gy + cell_h),
            "density": float(gv),
        })

heatmap_df = pl.DataFrame(heatmap_rows)
print(f"{heatmap_df.height} cells across {heatmap_df['attack_type'].n_unique()} classes "
      f"(threshold {HEATMAP_THRESHOLD}, vs theoretical max {100*100*len(data['density'])})")
heatmap_df.head()


3729 cells across 2 classes (threshold 0.05, vs theoretical max 20000)


attack_type,x,y,x2,y2,density
str,f64,f64,f64,f64,f64
"""Attack""",-11.011313,-1.765657,-10.676113,-1.506557,0.0511
"""Attack""",-11.688485,-1.503939,-11.353285,-1.244839,0.0511
"""Attack""",-11.349899,-1.503939,-11.014699,-1.244839,0.057
"""Attack""",-11.011313,-1.503939,-10.676113,-1.244839,0.0614
"""Attack""",-10.672727,-1.503939,-10.337527,-1.244839,0.0625


In [16]:
heatmap_chart = (
    alt.Chart(heatmap_df)
    .mark_rect()
    .encode(
        x=alt.X("x:Q", scale=alt.Scale(domain=[x1, x2]), axis=alt.Axis(title="UMAP 1")),
        x2="x2:Q",
        y=alt.Y("y:Q", scale=alt.Scale(domain=[y1, y2]), axis=alt.Axis(title="UMAP 2")),
        y2="y2:Q",
        color=alt.Color("attack_type:N", scale=color_scale, legend=alt.Legend(title=None, orient="top")),
        opacity=alt.condition(
            selection,
            alt.Opacity("density:Q", scale=alt.Scale(domain=[HEATMAP_THRESHOLD, 1.0], range=[0.03, 0.45]), legend=None),
            alt.value(0.0),
        ),
    )
)

main_c = (heatmap_chart + hull_chart + scatter).properties(width=400, height=400)
main_c


alt.LayerChart(...)

In [17]:
chart_c = (
    alt.vconcat(
        alt.hconcat(top_marginal, spacer, spacing=5),
        alt.hconcat(main_c, right_marginal, spacing=5),
        spacing=5,
    )
    .add_params(selection)
    .properties(
        title=alt.TitleParams(
            text="UMAP Projections of GAT Embeddings  ·  heatmap",
            subtitle=subtitle,
            anchor="start",
        )
    )
    .configure_view(stroke=None)
)

HTML_C = OUT_DIR / "umap_altair_heatmap.html"
SPEC_C = OUT_DIR / "umap_altair_heatmap.spec.json"
SVG_C  = OUT_DIR / "umap_altair_heatmap.svg"

chart_c.save(str(HTML_C))
SPEC_C.write_text(json.dumps(chart_c.to_dict(), indent=2))
SVG_C.write_text(vlc.vegalite_to_svg(json.dumps(chart_c.to_dict())))

print(f"HTML: {HTML_C.relative_to(REPO)}   {kb(HTML_C)}")
print(f"SPEC: {SPEC_C.relative_to(REPO)}   {kb(SPEC_C)}")
print(f"SVG : {SVG_C.relative_to(REPO)}   {kb(SVG_C)}")
chart_c


/tmp/ipykernel_24514/3851807859.py:8: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  .properties(


HTML: tools/spikes/umap_altair_heatmap.html     620.3 KB
SPEC: tools/spikes/umap_altair_heatmap.spec.json     897.1 KB
SVG : tools/spikes/umap_altair_heatmap.svg    1415.7 KB


alt.VConcatChart(...)

## 11. Variant D — stacked region-above-threshold polygons

The SveltePlot-faithful variant. Re-uses the exact polygons from Variant A (marching-squares on the normalized density grid), but renders them as `mark_area` instead of `mark_line` and draws them in ascending-level order. Each polygon outlines `{density >= level}`, not the band between levels; the stacked painter's-order opacity produces the gradient look without any `shapely.difference` step.

This is what `Plot.contour`, `d3-contour`, and SveltePlot's `<Contour>` do internally (per d3-contour source and Plot docs): there are no true annular bands, just stacked "region above threshold" polygons layered in ascending order. Variant B computes annular bands (matplotlib's approach); Variant D computes the Plot-style approach with **zero new data transforms** (same `contours_df` as Variant A, different mark).

**Size facts (verifiable):** D's interactive HTML and spec.json are the same size as Variant A because the polygon vertices are identical. `vl-convert`'s static SVG is much larger for D (~4,500 `<path>` elements vs 695 for A) because `mark_area` emits closed-polygon fills as many sub-paths. The interactive and static costs diverge — open both HTMLs to compare.


In [18]:
# Variant D main panel: stacked region-above-threshold polygons + hulls + scatter.
# Reuses `contours_df` from Variant A — same find_contours output, different mark.
filled_range = [0.05, 0.32]
filled_level_domain = [float(levels[0]), float(levels[-1])]

stacked_chart = (
    alt.Chart(contours_df)
    .mark_area(interpolate="linear")
    .encode(
        x=base_x,
        y=base_y,
        color=alt.Color("attack_type:N", scale=color_scale, legend=alt.Legend(title=None, orient="top")),
        detail="poly_id:N",
        order=alt.Order("order:Q", sort="ascending"),
        opacity=alt.condition(
            selection,
            alt.Opacity("level:Q", scale=alt.Scale(domain=filled_level_domain, range=filled_range), legend=None),
            alt.value(0.0),
        ),
    )
)

main_d = (stacked_chart + hull_chart + scatter).properties(width=400, height=400)
main_d


alt.LayerChart(...)

In [19]:
chart_d = (
    alt.vconcat(
        alt.hconcat(top_marginal, spacer, spacing=5),
        alt.hconcat(main_d, right_marginal, spacing=5),
        spacing=5,
    )
    .add_params(selection)
    .properties(
        title=alt.TitleParams(
            text="UMAP Projections of GAT Embeddings  ·  stacked polygons",
            subtitle=subtitle,
            anchor="start",
        )
    )
    .configure_view(stroke=None)
)

HTML_D = OUT_DIR / "umap_altair_stacked.html"
SPEC_D = OUT_DIR / "umap_altair_stacked.spec.json"
SVG_D  = OUT_DIR / "umap_altair_stacked.svg"

chart_d.save(str(HTML_D))
SPEC_D.write_text(json.dumps(chart_d.to_dict(), indent=2))
SVG_D.write_text(vlc.vegalite_to_svg(json.dumps(chart_d.to_dict())))

print(f"HTML: {HTML_D.relative_to(REPO)}   {kb(HTML_D)}")
print(f"SPEC: {SPEC_D.relative_to(REPO)}   {kb(SPEC_D)}")
print(f"SVG : {SVG_D.relative_to(REPO)}   {kb(SVG_D)}")
chart_d


/tmp/ipykernel_24514/1042913010.py:8: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  .properties(


HTML: tools/spikes/umap_altair_stacked.html     270.4 KB
SPEC: tools/spikes/umap_altair_stacked.spec.json     408.2 KB
SVG : tools/spikes/umap_altair_stacked.svg   33594.5 KB


alt.VConcatChart(...)

## 12. Size comparison — all four variants vs Svelte build

The incumbent `umap.html` bundle (from `make figures`) is the point of reference — it's self-contained JS+CSS via `vite-plugin-singlefile`. The Altair artifacts are `spec.json` alone; at render time the page loads Vega-Embed (one cached script, shared across every chart on the page).


In [20]:
SVELTE_BUILD = REPO / "_build" / "figures" / "umap.html"

def row(label: str, html: Path, spec: Path, svg: Path):
    h = html.stat().st_size / 1024 if html.exists() else 0.0
    s = spec.stat().st_size / 1024 if spec.exists() else 0.0
    v = svg.stat().st_size / 1024 if svg.exists() else 0.0
    print(f"{label:<28s} HTML {h:8.1f} KB   SPEC {s:8.1f} KB   SVG {v:8.1f} KB")

print("Altair variants")
print("-" * 80)
row("A  stroked isolines",  HTML_OUT, SPEC_OUT, SVG_OUT)
row("B  filled bands (shapely diff)", HTML_B, SPEC_B, SVG_B)
row("C  heatmap (rect)",    HTML_C,   SPEC_C,   SVG_C)
row("D  stacked polygons (Plot-like)", HTML_D, SPEC_D, SVG_D)

print()
if SVELTE_BUILD.exists():
    svelte_kb = SVELTE_BUILD.stat().st_size / 1024
    print(f"Svelte baseline (standalone, single-file bundle)   {svelte_kb:8.1f} KB")
else:
    print("run `make figures` first to compare against the Svelte build")


Altair variants
--------------------------------------------------------------------------------
A  stroked isolines          HTML    270.3 KB   SPEC    408.1 KB   SVG    223.6 KB
B  filled bands (shapely diff) HTML    294.7 KB   SPEC    441.6 KB   SVG  37260.5 KB
C  heatmap (rect)            HTML    620.3 KB   SPEC    897.1 KB   SVG   1415.7 KB
D  stacked polygons (Plot-like) HTML    270.4 KB   SPEC    408.2 KB   SVG  33594.5 KB

Svelte baseline (standalone, single-file bundle)      433.7 KB


## 13. Observations

**Check these after running:**

1. Does every HTML variant render with all classes visible at first load?
2. Click a legend entry on any variant — does it gray out one class across scatter, hulls, density layer, *and* marginals simultaneously?
3. Shift-click the legend — can you toggle multiple classes independently?
4. Does each SVG open and show all four panels (main + top marginal + right marginal + legend)?
5. Are the `spec.json` files reasonable? A and D (same polygon count) should be compact; B is slightly larger (Shapely-differenced annuli); C is heaviest (10k cells/class).

**The four variants at a glance (algorithm + verifiable size facts only — visual fidelity is for the reader to judge):**
- **A — stroked isolines.** `find_contours` + `mark_line`. Smallest spec and smallest static SVG.
- **B — filled bands (shapely).** `find_contours` → `shapely.difference` → `mark_area`. Produces annular polygons between adjacent levels (matplotlib `contourf` semantics). Spec is ~34 KB over A; `vl-convert` SVG is ~37 MB because each annulus becomes many `<path>` elements.
- **C — heatmap.** `mark_rect` on the raw 100×100 grid with opacity encoding. Most "Altair-native" — no marching-squares step at all. Biggest spec, medium SVG (~1.4 MB).
- **D — stacked polygons (Plot-like).** Same polygons as A, rendered as `mark_area` in ascending level order. This is the approach `Plot.contour`, `d3-contour`, and SveltePlot's `<Contour>` use internally (per d3-contour source). Interactive HTML and spec.json are identical in size to A; `vl-convert` static SVG is ~33 MB for the same mark-area expansion reason as B.

**Other trade-offs vs the Svelte version (mechanical, not aesthetic):**
- Toggle UI is legend-click, not buttons. Same semantics; removes custom state code.
- Metrics are in the subtitle, not a separate HTML row. Easy to move to a MyST caption cell if needed.

**Decision you have to make by opening files:**
- Which (if any) of the four actually looks acceptable? I cannot judge this.
- Are the panel proportions right (main : marginal ratio, aspect of the UMAP panel itself)? Currently hardcoded to 400×400 main, 400×60 top marginal, 60×400 right marginal — these numbers were picked blind and may need adjusting to match the data's x:y bound ratio or your aesthetic target.

If the variants need composition fixes (aspect ratio, marginal sizing, legend placement), please describe the specific issue — the fix lives in the compose cells (sections 7, 9–11) and can be retargeted once there's a concrete visual target.
